# Dataset Loading

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
import pandas as pd
from config.paths import STEAM_DATASET_PATH

In [12]:
def load_dataset():
    file_path = Path(STEAM_DATASET_PATH)

    df = pd.read_csv(
        file_path,
        header=None,
        names=["user_id", "game_title", "behavior", "value", "unused"]
    )

    return df

df = load_dataset()
df.head()

,user_id,game_title,behavior,value,unused
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0
1,151603712,The Elder Scrolls V Skyrim,play,273.0,0
2,151603712,Fallout 4,purchase,1.0,0
3,151603712,Fallout 4,play,87.0,0
4,151603712,Spore,purchase,1.0,0


# Data Preprocessing

This section applies filtering and cleaning steps to improve the quality of the graph representation.
The main goal is to reduce noise and keep only meaningful user–game interactions.

In [13]:
exact_duplicates = df[df.duplicated()]

print(f"Number of exact duplicate rows: {len(exact_duplicates)}")
exact_duplicates.head(10)

Number of exact duplicate rows: 707


,user_id,game_title,behavior,value,unused
1968,11373749,Sid Meier's Civilization IV,purchase,1.0,0
1970,11373749,Sid Meier's Civilization IV Beyond the Sword,purchase,1.0,0
1972,11373749,Sid Meier's Civilization IV Warlords,purchase,1.0,0
2724,56038151,Grand Theft Auto San Andreas,purchase,1.0,0
2726,56038151,Grand Theft Auto Vice City,purchase,1.0,0
2728,56038151,Grand Theft Auto III,purchase,1.0,0
3129,57103808,Sid Meier's Civilization IV,purchase,1.0,0
3556,148510973,Grand Theft Auto III,purchase,1.0,0
3562,148510973,Grand Theft Auto San Andreas,purchase,1.0,0
3575,148510973,Grand Theft Auto Vice City,purchase,1.0,0


In [14]:
# Store row counts throughout preprocessing
row_counts = []

# Initial dataset size
row_counts.append(("initial", len(df)))

# Remove the last column from the original dataset.
# This column contains only constant values (0) and does not add any useful information.
df = df.drop(columns=["unused"])
row_counts.append(("drop_unused_column", len(df)))

# Keep only rows where the user actually played the game.
# The 'purchase' interaction only indicates ownership, while 'play' gives us usage behavior.
df_play = df[df["behavior"] == "play"].copy()
row_counts.append(("keep_play_only", len(df_play)))

# Rename value column to improve readability
df_play = df_play.rename(columns={"value": "hours_played"})

# Convert the playtime column to numeric values.
# This ensures we can apply numerical filters and statistics later.
df_play["hours_played"] = pd.to_numeric(df_play["hours_played"])

# Remove duplicated user-game interactions
# Ensures each user contributes only once per game
df_play = df_play.drop_duplicates(
    subset=["user_id", "game_title"]
)
row_counts.append(("remove_duplicate_user_game", len(df_play)))

# Remove interactions with less than 1 hour of playtime.
# This helps reduce noise from games that were opened only briefly.
df_play = df_play[df_play["hours_played"] >= 1]
row_counts.append(("hours_played >= 1", len(df_play)))

# Keep only users who have played at least 2 different games.
# Users with a single game do not contribute to similarity relationships.
valid_users = df_play["user_id"].value_counts()
valid_users = valid_users[valid_users >= 2].index
df_play = df_play[df_play["user_id"].isin(valid_users)]
row_counts.append(("users >= 2 games", len(df_play)))

# Keep only games played by at least 10 users.
# This removes very rare games that would create isolated nodes or weak edges.
valid_games = df_play["game_title"].value_counts()
valid_games = valid_games[valid_games >= 10].index
df_play = df_play[df_play["game_title"].isin(valid_games)]
row_counts.append(("games >= 10 players", len(df_play)))

In [15]:
preprocessing_summary = pd.DataFrame(
    row_counts,
    columns=["step", "rows"]
)

preprocessing_summary

,step,rows
0,initial,200000
1,drop_unused_column,200000
2,keep_play_only,70489
3,remove_duplicate_user_game,70477
4,hours_played >= 1,53694
5,users >= 2 games,48183
6,games >= 10 players,41948


# Graph Construction

# Initial Analysis

# Export